# NUKÚN × Akwɛ — validation GPU

Charger l’archive locale générée par `python scripts/package_source.py`. Sélectionner un runtime GPU avant exécution. Ce notebook relance CPU et GPU sur le même hôte, conserve les deux résultats et refuse de revendiquer CUDA sans GPU. Aucune donnée personnelle n’est utilisée.


In [ ]:
from google.colab import files
from pathlib import Path
from zipfile import ZipFile
import hashlib, json, os, subprocess, sys

uploaded = files.upload()
archive_path = next(Path(name) for name in uploaded if name.endswith('.zip'))
destination = Path('/content').resolve()
with ZipFile(archive_path) as archive:
    for item in archive.infolist():
        if not (destination / item.filename).resolve().is_relative_to(destination):
            raise ValueError('Chemin d’archive invalide')
    archive.extractall(destination)
project = destination / 'nukun-akwe'
manifest = json.loads((project / 'SOURCE_MANIFEST.json').read_text())
for item in manifest['files']:
    assert hashlib.sha256((project / item['path']).read_bytes()).hexdigest() == item['sha256'], item['path']
os.chdir(project)
print('Archive vérifiée :', len(manifest['files']), 'fichiers')


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt', '-r', 'requirements-graph.txt'], check=True)
import torch
print('PyTorch :', torch.__version__, 'CUDA :', torch.version.cuda)
assert torch.cuda.is_available(), 'CUDA indisponible : choisir un runtime GPU et un build PyTorch compatible.'
print('GPU :', torch.cuda.get_device_name(0))
print('VRAM :', torch.cuda.get_device_properties(0).total_memory)
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_stress.py', '-v'], check=True)


In [ ]:
# Même protocole et même machine, deux répertoires distincts.
for device, output in [('cpu', 'runtime/cpu_same_host'), ('cuda', 'runtime/gpu_same_host')]:
    subprocess.run([sys.executable, '-m', 'nukun.stress_experiment', '--device', device, '--output-root', output], check=True)


In [ ]:
import csv
cpu = json.loads(Path('runtime/cpu_same_host/reports/stress_metrics.json').read_text())
gpu = json.loads(Path('runtime/gpu_same_host/reports/stress_metrics.json').read_text())
assert cpu['protocol_sha256'] == gpu['protocol_sha256']
assert cpu['sources'] == gpu['sources']
assert gpu['environment']['device'] == 'cuda' and gpu['environment']['gpu_name']
comparison = []
for left, right in zip(cpu['runs'], gpu['runs'], strict=True):
    assert left['seed'] == right['seed'] and left['dataset_sha256'] == right['dataset_sha256']
    for name in ['mlp_history', 'mlp_akwe', 'sage_history', 'sage_akwe']:
        a, b = left['training'][name], right['training'][name]
        comparison.append({'seed': left['seed'], 'model': name, 'cpu_seconds': a['seconds'], 'gpu_seconds': b['seconds'], 'cpu_over_gpu': a['seconds']/b['seconds'], 'peak_vram_bytes': b['peak_vram_bytes']})
with Path('runtime/comparison.csv').open('w', newline='') as stream:
    writer = csv.DictWriter(stream, fieldnames=list(comparison[0]))
    writer.writeheader()
    writer.writerows(comparison)
comparison


In [ ]:
from zipfile import ZIP_DEFLATED
bundle = Path('/content/nukun-gpu-results.zip')
with ZipFile(bundle, 'w', compression=ZIP_DEFLATED) as archive:
    for directory in ['runtime/cpu_same_host', 'runtime/gpu_same_host']:
        for path in Path(directory).rglob('*'):
            if path.is_file():
                archive.write(path, path.as_posix())
    archive.write('runtime/comparison.csv', 'comparison.csv')
files.download(str(bundle))


## Interpréter les résultats

Un ratio CPU/GPU inférieur à 1 indique une exécution GPU plus lente sur cette charge. Les timings d’entraînement excluent la préparation des tenseurs. Conserver matériel, versions, VRAM et métriques des deux appareils ; ne pas extrapoler les résultats de simulation à des fraudes réelles. Le notebook est préparé localement ; son exécution distante reste à faire.
